# Cookie Cats player retention

A retrospective reproduction of a public experiment. Dataset and reference analysis are credited in [ATTRIBUTION.md](../ATTRIBUTION.md). This notebook uses this repository's analysis functions so calculations and command-line results stay consistent.

**Question:** should the first gate move from level 30 to level 40? Day 7 retention is the primary metric for this reproduction; Day 1 is supporting evidence. This is not a preregistered analysis plan.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'analyze.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from analyze import load_players, analyze, markdown_report, plot_results


## Validate the population

Keep all assigned players in the main analysis. Reject missing values, duplicate IDs, unexpected groups, and invalid flags. Run `python download_data.py` from the repository root before starting.

In [2]:
players = load_players(ROOT / 'data' / 'cookie_cats.csv')
print(f'Players: {len(players):,}')
print(players.groupby('version').agg(players=('userid', 'size'), day_1=('retention_1', 'mean'), day_7=('retention_7', 'mean')).to_string())
print('Zero-round players:', int(players.sum_gamerounds.eq(0).sum()))
print('Maximum game rounds:', int(players.sum_gamerounds.max()))


Players: 90,189
         players     day_1     day_7
version                             
gate_30    44700  0.448188  0.190201
gate_40    45489  0.442283  0.182000
Zero-round players: 3994
Maximum game rounds: 49854


## Estimate retention differences

The pooled two-proportion test evaluates equal retention rates. The confidence interval uses separate group variances. For binary outcomes, a binomial draw at the empirical rate exactly reproduces within-group bootstrap resampling. Differences are gate 40 minus gate 30.

In [3]:
results, samples = analyze(players, iterations=10_000, seed=42)
print(markdown_report(results))


# Cookie Cats analysis results

Retrospective reproduction; differences are gate 40 minus gate 30. Every assigned player is included in the primary analysis.

| Metric | Gate 30 | Gate 40 | Difference | 95% normal CI | p-value |
|---|---:|---:|---:|---:|---:|
| retention_1 | 44.82% | 44.23% | -0.59 pp | -1.24 to +0.06 pp | 0.0744 |
| retention_7 | 19.02% | 18.20% | -0.82 pp | -1.33 to -0.31 pp | 0.0016 |

Allocation check, assuming 50/50: p = 0.0086. This needs assignment logs to investigate, and does not prove faulty randomization.

## Sensitivity checks

- without_max_rounds: Day 7 difference -0.82 pp, p = 0.0016.
- positive_rounds_only: Day 7 difference -0.81 pp, p = 0.0026.

## Recommendation

Keep the gate at level 30 pending investigation of allocation and revenue trade-offs.

## Interpretation limits

- Day 7 is the decision metric for this reproduction; no historical preregistration is claimed.
- Day 1 is supporting evidence. A non-significant result does not establish equivale

## Generate the figures

The Day 7 bootstrap differences are mainly below zero. The fraction below zero describes these resamples, not the probability that the true treatment effect is negative. Game rounds are shown on a log scale because of their long right tail.

In [4]:
plot_results(players, results, samples, ROOT / 'reports' / 'figures')
for metric, item in results['retention'].items():
    print(metric, 'bootstrap 95% CI (pp):', item['bootstrap_ci_95_pp'])


retention_1 bootstrap 95% CI (pp): [-1.237976287799348, 0.055905362325958656]
retention_7 bootstrap 95% CI (pp): [-1.3187317023763108, -0.3130621851052997]


![Bootstrap retention differences](../reports/figures/retention_bootstrap.png)

![Game rounds](../reports/figures/game_rounds.png)

## Decision and caveats

Keep level 30 pending investigation of allocation and revenue. The 50/50 assignment diagnostic flags an imbalance; assignment logs are unavailable. Removing the maximum-round player and excluding zero-round players leave the observed direction unchanged, but conditioning on later engagement can bias causal comparisons. No revenue or gate-exposure data are available. The Day 1 result does not establish equivalence.